In [0]:
# Bronze: raw Delta -> flatten EIA response.data -> MERGE (incremental, idempotent)
from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", "UTC")

CATALOG      = "sample_project_playground_003"
RAW_TABLE    = f"{CATALOG}.raw.eia_raw_json"
BRONZE_TABLE = f"{CATALOG}.bronze.eia_natural_gas_prices"
CKPT         = f"/Volumes/{CATALOG}/raw/checkpoints/eia_bronze_prices"

# --- DDL (idempotent) -------------------------------------------------------
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {BRONZE_TABLE} (
  series             STRING,
  series_description STRING,
  product            STRING,
  product_name       STRING,
  process            STRING,
  process_name       STRING,
  duoarea            STRING,
  area_name          STRING,
  period             DATE,
  price_usd_mmbtu    DOUBLE,
  units              STRING,
  frequency          STRING,
  api_version        STRING,
  source_dataset     STRING,
  source_file_path   STRING,
  bronze_ingested_at TIMESTAMP
)
USING DELTA
CLUSTER BY (series)
""")

COLS = ["series", "series_description", "product", "product_name",
        "process", "process_name", "duoarea", "area_name", "period",
        "price_usd_mmbtu", "units", "frequency", "api_version",
        "source_dataset", "source_file_path", "bronze_ingested_at"]

# --- EIA v2 document schema (backticks: hyphenated keys are not identifiers) -
PAYLOAD_SCHEMA = """
STRUCT<
  apiVersion: STRING,
  response: STRUCT<
    total: STRING,
    dateFormat: STRING,
    frequency: STRING,
    data: ARRAY<
      STRUCT<
        period: STRING,
        duoarea: STRING,
        `area-name`: STRING,
        product: STRING,
        `product-name`: STRING,
        process: STRING,
        `process-name`: STRING,
        series: STRING,
        `series-description`: STRING,
        value: STRING,
        units: STRING
      >
    >
  >
>
"""

def build_bronze(df):
    rows = (
        df.select(
            F.col("payload").cast("string").alias("json_str"),
            F.col("source_folder"),
            F.col("file_path").alias("source_file_path"),
        )
        .withColumn("doc", F.from_json(
            F.regexp_replace(F.col("json_str"), "^\ufeff", ""),  # defensive BOM strip
            PAYLOAD_SCHEMA))
        .select("source_folder", "source_file_path",
                F.col("doc.apiVersion").alias("api_version"),
                F.col("doc.response.frequency").alias("frequency"),
                F.explode("doc.response.data").alias("r"))
    )

    # hyphenated struct fields need getField(); value lives inside r too
    f = lambda name: F.col("r").getField(name)

    return rows.select(
        f("series").alias("series"),
        f("series-description").alias("series_description"),
        f("product").alias("product"),
        f("product-name").alias("product_name"),
        f("process").alias("process"),
        f("process-name").alias("process_name"),
        f("duoarea").alias("duoarea"),
        f("area-name").alias("area_name"),
        F.to_date(f("period")).alias("period"),
        f("value").cast("double").alias("price_usd_mmbtu"),
        f("units").alias("units"),
        F.col("frequency"),
        F.col("api_version"),
        F.col("source_folder").alias("source_dataset"),
        F.col("source_file_path"),
        F.current_timestamp().alias("bronze_ingested_at"),
    ).where(
        F.col("series").isNotNull() & F.col("period").isNotNull()
    )

def merge_bronze(batch_df, batch_id):
    n_raw = batch_df.count()
    staged = build_bronze(batch_df).cache()

    if n_raw > 0 and staged.isEmpty():
        staged.unpersist()
        raise RuntimeError(f"batch {batch_id}: {n_raw} raw rows but 0 bronze rows - parse failure")

    if staged.isEmpty():
        staged.unpersist()
        return

    sel = ", ".join(COLS)
    staged.createOrReplaceTempView("stg")
    null_prices = spark.sql(
        "SELECT count(*) AS n FROM stg WHERE price_usd_mmbtu IS NULL"
    ).collect()[0]["n"]
    print(f"batch {batch_id}: raw={n_raw}, staged={staged.count()}, null_price={null_prices}")

    # in-batch dedup (daily overlaps historical) -> later ingest wins, MERGE safe
    spark.sql(f"""
      MERGE INTO {BRONZE_TABLE} t
      USING (
        SELECT {sel}
        FROM (
          SELECT *, ROW_NUMBER() OVER (
            PARTITION BY series, period
            ORDER BY bronze_ingested_at DESC, source_file_path DESC
          ) AS rn
          FROM stg
        )
        WHERE rn = 1
      ) s
      ON  t.series = s.series
      AND t.period = s.period
      WHEN MATCHED THEN UPDATE SET *
      WHEN NOT MATCHED THEN INSERT *
    """)
    staged.unpersist()

# --- Run: incremental via checkpoint ----------------------------------------
query = (
    spark.readStream.format("delta")
    .table(RAW_TABLE)
    .writeStream
    .foreachBatch(merge_bronze)
    .option("checkpointLocation", CKPT)
    .trigger(availableNow=True)
    .start()
)
query.awaitTermination()

# --- Bronze DQ ---------------------------------------------------------------
dups = spark.sql(f"""
  SELECT count(*) AS n FROM (
    SELECT 1 FROM {BRONZE_TABLE}
    GROUP BY series, period
    HAVING count(*) > 1)
""").collect()[0]["n"]
assert dups == 0, f"{dups} duplicate keys in bronze"

# reported, not failed: EIA may legitimately return rows without a value
cast_fail = spark.sql(f"""
  SELECT count(*) AS n FROM {BRONZE_TABLE}
  WHERE price_usd_mmbtu IS NULL
""").collect()[0]["n"]
print(f"null-price rows (EIA gaps, reported not failed): {cast_fail}")

spark.sql(f"""
  SELECT count(*)                AS rows_total,
         count(DISTINCT series)  AS series_cnt,
         min(period)             AS first_day,
         max(period)             AS last_day,
         min(price_usd_mmbtu)    AS min_price,
         max(price_usd_mmbtu)    AS max_price
  FROM {BRONZE_TABLE}
""").show()

spark.sql(f"""
  SELECT series, process_name, count(*) AS rows, max(period) AS last_day
  FROM {BRONZE_TABLE}
  GROUP BY 1, 2 ORDER BY 1
""").show()